# 02 - Train & EDA
Person 2. EDA + feature engineering + model training on train.csv.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')
from utils import TARGET

train_df = pd.read_csv('../data/train.csv')
train_df.head()


## EDA - target balance

In [ ]:
sns.countplot(x=TARGET, data=train_df)
plt.title('Attrition Balance (Train Set)')
plt.savefig('../outputs/charts/attrition_balance.png', bbox_inches='tight')
plt.show()


## EDA - attrition by key categorical features
Adjust column names below to match your dataset.

In [ ]:
cat_features = [c for c in ['Department', 'JobRole', 'OverTime', 'MaritalStatus'] if c in train_df.columns]

for col in cat_features:
    plt.figure()
    sns.countplot(x=col, hue=TARGET, data=train_df)
    plt.title(f'Attrition by {col}')
    plt.xticks(rotation=45)
    plt.savefig(f'../outputs/charts/attrition_by_{col}.png', bbox_inches='tight')
    plt.show()


## EDA - correlation heatmap (numeric features)

In [ ]:
num_df = train_df.select_dtypes(include=np.number)
plt.figure(figsize=(10, 8))
sns.heatmap(num_df.corr(), cmap='coolwarm', center=0)
plt.title('Numeric Feature Correlation')
plt.savefig('../outputs/charts/correlation_heatmap.png', bbox_inches='tight')
plt.show()


## Feature engineering / encoding

In [ ]:
df = train_df.copy()
df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0})

cat_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X_train = df.drop(columns=[TARGET])
y_train = df[TARGET]
X_train.head()


## Train model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

joblib.dump(model, '../model/attrition_model.pkl')
joblib.dump(list(X_train.columns), '../model/model_columns.pkl')  # needed later to align test/present sets
print('model saved')


## Feature importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 10 Feature Importances')
plt.savefig('../outputs/charts/feature_importance.png', bbox_inches='tight')
plt.show()
